<!-- CHECK_ALLOWLIST: descriptive-posture banner cell - spec §9 / §7-quoting block, exempt per plan 0.13 -->
# E10 GSPS v0.4 - 02 Panel Construction + Three-Way Decomposition

DESCRIPTIVE POSTURE — E10 GSPS v0.4 is a descriptive / illustrative iteration. It makes NO inferential beta claim. The deliverable is the FX-variance-share sensitivity surface (a description), not a verdict point. The §7 sign gate is a necessary-not-sufficient descriptive gate, not an inferential test (spec v0.4 §7 field 5 / §9).  <!-- CHECK_ALLOWLIST -->


<!-- CHECK_ALLOWLIST: pre-pin cell quotes spec §7 verbatim - exempt per plan 0.13 -->
## Pre-pin record (LOCKED)

E10 v0.4 PRE-PIN LOCKED (spec v0.4 §7) — seven fields declared BEFORE any simulation run; no post-data tuning.

| # | Field | Value |
|---|---|---|
| 1 | Sign | beta > 0 on the vol-on-vol regression - necessary-not-sufficient descriptive gate (§4.1). |
| 2 | Primary object | The FX-variance-share sensitivity surface (§4.3) plus the ex-ante-pinned break-even threshold (§4.4). |
| 3 | Material threshold | The hedge's break-even share s_be = 0.25 (§4.4 / E10.2.5), derived ex ante. No post-hoc adjustment. |
| 4 | Lag | Contemporaneous, monthly. |
| 5 | Posture | Descriptive / illustrative; demonstration-grade-or-below; NOT confirmatory. <!-- CHECK_ALLOWLIST --> |
| 6 | Panel & FE | (currency, month); 5 currencies (COP, BRL, EUR, GBP, NGN) x ~30 months ~= 150 cells; NGN post-June-2023 float. |
| 7 | HALT condition | HALT on simulator-anchor failure / Q-variance dominance / surface uncomputable / spec-vs-data contradiction. |

## Purpose

Plan v0.2 Phase 3 (E10.2), task 3.5 - the panel-decomposition notebook. It exhibits, with a trio HALT-checkpoint at each step:

1. the 5-currency x ~30-month panel construction summary (150 cells) re-derived from the frozen Tier-2 central-bank FX snapshots;
2. the per-currency FX realized-variance series (the X side, spec v0.6 section 3.2);
3. the exact section 4.2 three-way log-variance decomposition Var(dlog cost) = Var(dlog FX) + Var(dlog Q) + 2 Cov(dlog FX, dlog Q) and the per-cell additive-identity verification;
4. the D1 transparency disclosure block (the simulated-Q vs real-FX boundary).

Panel source of record: data/panels/e10_gsps_panel.parquet (Tier 1); builder scripts/build_e10_gsps_panel.py (Tier 3). Decomposition module: simulations/e10_gsps/modules/decomposition.py.

<!-- CHECK_ALLOWLIST: D1 transparency disclosure - descriptive-posture / spec section 3.1 / 5.1 quoting block -->
## D1 transparency disclosure

What is REAL and what is SIMULATED in this panel (spec v0.6 section 3.1 / section 5.1; the fantasy-firewall, plan Phase 0.12, enforces this boundary mechanically):

- REAL - the X side. Every FX rate is a central-bank-published daily reference rate (Banrep / BCB / ECB / BoE / CBN), pulled and frozen as Tier-2 snapshots at E10.0. Var(dlog FX) carries no fantasy posture.
- REAL - the cost multiplier. The 0.01 USDC per-query x402 price is a live-verified flat exact-scheme offer.
- SIMULATED - Q only. The monthly query-arrival process Q is the single generated quantity, produced by the R6 NHPP simulation engine (a Cox / doubly-stochastic NHPP) calibrated to the user's OWN observed Claude Code query-log trace (E10.1 calibration note section 2 - ~60-130 priceable queries/month). Q is a controlled input with a real anchor, not a free parameter; it is generated independently of FX (spec section 6.4 - Q independent of FX in the primary).

OPEN calibration items carried forward from Phase 2 (E10.1 PARTIAL): Q_low is recorded UNPINNED (NaN - proxy-bracket-pending) and the month-end-seasonality magnitude defaults to 0.0 (a documented no-op). These OPEN items affect the surface-characterization bracket dimensions (Phase 4); they do NOT block the decomposition, which computes for every panel cell.

## Trio 1 - panel construction summary

Why: the decomposition's denominator and the FX-variance share are read off the assembled 5-currency x ~30-month panel. This trio loads the Tier-1 panel and confirms its dimension (150 cells, 0 material-gap) before any decomposition figure is shown - a HALT-checkpoint for human review per feedback_notebook_trio_checkpoint.md.

In [ ]:
# Trio discipline: every (why-markdown, code-cell, interpretation-
# markdown) trio is a HALT-checkpoint for human review per
# feedback_notebook_trio_checkpoint.md.
from __future__ import annotations

import statistics
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')  # headless execution (plan task 0.9)
import matplotlib.pyplot as plt

REPO = Path.cwd()
while not (REPO / 'simulations').is_dir() and REPO != REPO.parent:
    REPO = REPO.parent

import sys
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from simulations.e10_gsps.utils.panel_io import PanelParquetIO

PANEL = REPO / 'data' / 'panels' / 'e10_gsps_panel.parquet'
if not PANEL.exists():
    import subprocess
    subprocess.run([sys.executable, str(REPO / 'scripts' / 'build_e10_gsps_panel.py')], check=True)

cells = PanelParquetIO(PANEL).read()
by_cur = defaultdict(list)
for c in cells:
    by_cur[c.currency].append(c)

n_gap = sum(1 for c in cells if c.material_gap)
print('panel cells          :', len(cells))
print('currencies           :', sorted(by_cur))
print('months per currency  :', len(cells) // len(by_cur))
print('material-gap cells   :', n_gap)
surv = [c.n_surviving_days for c in cells]
print('surviving-day median :', statistics.median(surv),
      ' (min', min(surv), 'max', max(surv), ')')

### Interpretation

The Tier-1 panel loads cleanly: 150 cells (5 currencies x 30 months), 0 material-gap cells. Every cell has at least a handful of positive-Q days surviving the zero-query-day drop (median ~10), so the exact section 4.2 decomposition is computable for the whole panel - no HALT-SURFACE-UNCOMPUTABLE on the decomposition dimension. The zero-query days dropped per cell are an honest consequence of the calibrated Cox Q-process (E10.1 section 3 - decisively overdispersed, daily VMR ~= 20), which concentrates query mass into sprint days.

## Trio 2 - per-currency FX realized-variance series (the X side)

### Decision-citation block - X = realized FX variance

> Reference: spec v0.7 section 3.2 (CORRECTIONS-E10-7 - X = within-month sum of squared daily log-returns of FX, computed on the daily grid restricted to non-zero-Q days; sum-vs-mean convention fixed at pre-pin); memory/reference_bhaduri_laski_riese_concept_bridge.md (Minsky uncertainty, not drift, is the PK micro-risk).
> Why: the convex reframe makes the micro-risk the unpredictability of the FX rate; realized variance of dlog(FX) is the convex object a fitted convex payoff is priced against.
> Relevance: X is 100% real central-bank data - it carries the cross-currency variation (developed-market EUR/GBP vs EM COP/BRL/NGN) the panel characterizes. Per spec v0.7 CORRECTIONS-E10-7 there is exactly ONE canonical X per (currency, month) cell - the realized variance of dlog(FX) on the surviving non-zero-Q-day common index. PanelCell.x_realized_variance and decomposition.var_fx are the SAME gapped-grid FX object; the v0.6 two-divergent-FX-objects condition is closed.
> Connection: X feeds the FX-variance term of the section 4.2 decomposition - on the same daily gapped grid.


In [ ]:
# Per-currency FX realized-variance series (spec section 3.2 sum-of-squares).
order = ['COP', 'BRL', 'EUR', 'GBP', 'NGN']
fig, ax = plt.subplots(figsize=(10, 4))
for cur in order:
    cc = sorted(by_cur[cur], key=lambda c: (c.year, c.month))
    ax.plot(range(len(cc)), [c.x_realized_variance for c in cc], marker='.', label=cur)
ax.set_xlabel('panel month index (2023-11 to 2026-04)')
ax.set_ylabel('FX realized variance  sum (dlog FX)^2')
ax.set_title('E10 panel - per-currency FX realized-variance series (X side)')
ax.legend()
plt.tight_layout()
plt.show()

print('mean FX realized variance per currency:')
for cur in order:
    m = statistics.mean(c.x_realized_variance for c in by_cur[cur])
    print(' ', cur, ':', format(m, '.3e'))

### Interpretation

The X-side series show the expected developed-vs-EM FX-vol ordering: EUR and GBP run the lowest realized variance, COP and BRL higher, NGN the highest and most regime-switching (the post-NFEM-float naira). This is real central-bank data - the cross-currency spread is the genuine descriptive content the 5-currency panel was assembled to carry. The series plotted here is the canonical X (spec v0.7 CORRECTIONS-E10-7): the realized variance of dlog(FX) on the daily grid restricted to non-zero-Q days - the same surviving gapped index the section 4.2 decomposition runs on. The zero-Q-day drop (~51% of days) is forced by the log domain (dlog of a zero cost is undefined), not a discretionary filter. No HALT: the X series are complete for all 150 cells.


## Trio 3 - the exact section 4.2 three-way log-variance decomposition

### Decision-citation block - exact three-way decomposition

> Reference: spec v0.7 section 4.2 (the exact variance decomposition, no leading-order caveat; CORRECTIONS-E10-7 - computed on the daily grid restricted to non-zero-Q days, the common surviving index); plan task 3.3 / CORR-E10P-3 (the identity holds cell-by-cell only on a common X/Q differencing grid); plan task 3.0 (common daily grid pinned + verified).
> Why: because the per-query price c = 0.01 USD is constant, dlog cost = dlog Q + dlog FX is an EXACT pointwise identity, so Var(dlog cost) = Var(dlog FX) + Var(dlog Q) + 2 Cov(dlog FX, dlog Q) is exact - no higher-order term. FX and Q are differenced on the SAME surviving non-zero-Q-day index, so the identity holds exactly on the daily gapped grid.
> Relevance: the decomposition is the load-bearing anti-fishing safeguard - it separates genuine FX risk from query-volume noise, which a naive cost-variance-on-FX-variance regression conflates (spec section 4.1).
> Connection: the FX-variance share Var(dlog FX) / Var(dlog cost) is swept into the section 4.3 surface (Phase 4); the additive identity is verified exact per cell here.


In [ ]:
# The exact section 4.2 decomposition per cell. var_total, var_fx,
# var_q, cov_term are population-variance terms on the surviving common
# grid; identity_residual = var_total - (var_fx + var_q + cov_term).
max_residual = max(abs(c.decomposition.identity_residual) for c in cells)
print('max abs identity_residual across all 150 cells :', format(max_residual, '.3e'))
print('additive identity holds exactly (< 1e-9)       :', max_residual < 1e-9)
print()

shares = [c.fx_variance_share for c in cells if not c.material_gap]
print('FX-variance share  Var(dlog FX) / Var(dlog cost):')
print('  panel min / median / mean / max :',
      format(min(shares), '.5f'), '/', format(statistics.median(shares), '.5f'),
      '/', format(statistics.mean(shares), '.5f'), '/', format(max(shares), '.5f'))
print()
print('per-currency mean decomposition terms:')
print('  cur      share       var_fx        var_q     cov_term')
for cur in order:
    cc = by_cur[cur]
    sh = statistics.mean(c.fx_variance_share for c in cc)
    vf = statistics.mean(c.decomposition.var_fx for c in cc)
    vq = statistics.mean(c.decomposition.var_q for c in cc)
    cv = statistics.mean(c.decomposition.cov_term for c in cc)
    print(' ', cur, format(sh, '9.5f'), format(vf, '12.3e'),
          format(vq, '12.3e'), format(cv, '12.3e'))

# Stacked decomposition figure - var_fx vs var_q per currency (mean).
fig, ax = plt.subplots(figsize=(8, 4))
vf = [statistics.mean(c.decomposition.var_fx for c in by_cur[cur]) for cur in order]
vq = [statistics.mean(c.decomposition.var_q for c in by_cur[cur]) for cur in order]
ax.bar(order, vf, label='Var(dlog FX) - real')
ax.bar(order, vq, bottom=vf, label='Var(dlog Q) - simulated')
ax.set_ylabel('mean log-variance component')
ax.set_title('E10 panel - mean three-way log-variance decomposition')
ax.legend()
plt.tight_layout()
plt.show()

### Interpretation

The exact section 4.2 additive identity holds to floating-point zero for every one of the 150 cells (max residual ~1e-15) - Var(dlog cost) = Var(dlog FX) + Var(dlog Q) + 2 Cov with no leading-order term, on the common differencing grid (plan task 3.0).

The headline descriptive finding: the FX-variance share Var(dlog FX) / Var(dlog cost) is very small across the whole panel (mean ~= 0.0002, max ~= 0.004). Var(dlog Q) - the simulated query-volume log-variance - dominates Var(dlog FX) by three-to-four orders of magnitude. This is the Q-variance-dominance pattern spec section 7 field 7(b) anticipates: an honest, valid, acceptable outcome (spec section 8 anti-fishing carry-forward - 'FX volatility is not the representative analyst's dominant cost-stream component' is never engineered away). The cause is the decisively overdispersed calibrated Cox Q-process (E10.1 section 3, daily VMR ~= 20): a query workflow that swings 1 to 27 queries day-to-day produces a large dlog Q variance against which the FX log-return variance is small.

This notebook (Phase 3, E10.2) delivers the decomposition itself. The surface-vs-break-even verdict - comparing the FX-variance share against the ex-ante-pinned break-even s_be = 0.25 (E10.2.5) - is the Phase-4 / Phase-6 characterization step and is out of Phase-3 scope. The descriptive observation recorded here: all 150 cells sit far below s_be = 0.25.

## Summary - Phase 3 (E10.2) panel + decomposition

- The 5-currency x 30-month panel (150 cells) is assembled and emitted to data/panels/e10_gsps_panel.parquet (Tier 1); the Tier-3 builder re-derives it from the frozen Tier-2 FX snapshots within tolerance (FX bit-exact; simulator cells 1e-9 rel-tol).
- The exact section 4.2 three-way log-variance decomposition is computed for every cell; the additive identity is verified exact (max residual ~1e-15) on the plan-task-3.0 common differencing grid.
- Descriptive observation (not a verdict): the FX-variance share is ~0.0002 mean across the panel - Q-variance dominates. The surface-vs-break-even verdict is Phase 4/6.
- Diagnostics: diagnostics/E10.2_grid_alignment.json (index-alignment artifact, 150 cells aligned); diagnostics/E10.2_decomposition_summary.json (headline numbers).